# Social Media Advertising Analysis  
## Senior Data Scientist + AI Engineer Style Notebook

This notebook analyzes a **Social Media Advertising dataset** for marketing and customer insights.

### Project Goals
- Understand campaign performance
- Clean and prepare marketing data
- Create important marketing KPIs
- Analyze channels, customer segments, campaign goals, locations, and languages
- Build simple visualizations
- Create campaign performance segments using K-Means
- Build a simple AI/ML model to predict **High ROI campaigns**

### Dataset
File used: `Social_Media_Advertising.csv`

---

# Phase 1: Import Required Libraries

We use simple and powerful Python libraries:

- `pandas` for data analysis
- `numpy` for numerical calculations
- `matplotlib` for charts
- `scikit-learn` for machine learning and clustering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

%matplotlib inline

# Phase 2: Load Dataset

Run this cell after placing `Social_Media_Advertising.csv` in the same folder as this notebook.

If you are running inside ChatGPT sandbox, the fallback path is also included.

In [ ]:
DATA_PATH = Path("Social_Media_Advertising.csv")

# Fallback path for ChatGPT / sandbox environment
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/Social_Media_Advertising.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

# Phase 3: Basic Data Understanding

As a senior analyst, first we check:

- Number of rows and columns
- Column names
- Data types
- Missing values
- Duplicate rows
- Basic statistics

In [ ]:
# Dataset shape
df.shape

In [ ]:
# Column names
df.columns.tolist()

In [ ]:
# Data types
df.info()

In [ ]:
# Missing values
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values

In [ ]:
# Duplicate rows
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

In [ ]:
# Basic statistical summary
df.describe(include="all").T

## Business Meaning of Important Columns

| Column | Meaning |
|---|---|
| `Campaign_ID` | Unique campaign ID |
| `Target_Audience` | Audience group targeted by campaign |
| `Campaign_Goal` | Objective of campaign |
| `Duration` | Campaign duration |
| `Channel_Used` | Marketing channel used |
| `Conversion_Rate` | Percentage/rate of users converted |
| `Acquisition_Cost` | Cost spent to acquire customers |
| `ROI` | Return on Investment |
| `Clicks` | Number of clicks |
| `Impressions` | Number of ad impressions |
| `Engagement_Score` | Engagement quality score |
| `Customer_Segment` | Customer type |
| `Date` | Campaign date |
| `Company` | Company/brand name |

---

# Phase 4: Data Cleaning and Feature Engineering

We will create clean and useful columns:

- Convert `Date` into datetime format
- Convert `Acquisition_Cost` from text like `$500.00` into numeric value
- Create `Month` and `Year`
- Create marketing KPIs:
  - CTR %
  - Estimated Conversions
  - CPC
  - CPA
  - Estimated Revenue

### Important Assumption
The dataset already contains `ROI`. Here we estimate revenue using:

`Estimated Revenue = Acquisition Cost × (1 + ROI)`

This is a practical business assumption for analysis.

In [ ]:
df_clean = df.copy()

# Convert Date column
df_clean["Date"] = pd.to_datetime(df_clean["Date"], errors="coerce")

# Clean Acquisition_Cost column
df_clean["Acquisition_Cost_Clean"] = (
    df_clean["Acquisition_Cost"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# Date features
df_clean["Year"] = df_clean["Date"].dt.year
df_clean["Month"] = df_clean["Date"].dt.month
df_clean["Month_Name"] = df_clean["Date"].dt.month_name()
df_clean["Year_Month"] = df_clean["Date"].dt.to_period("M").astype(str)

# Marketing KPIs
df_clean["CTR_Percent"] = (df_clean["Clicks"] / df_clean["Impressions"]) * 100
df_clean["Estimated_Conversions"] = df_clean["Clicks"] * df_clean["Conversion_Rate"]
df_clean["CPC"] = df_clean["Acquisition_Cost_Clean"] / df_clean["Clicks"]
df_clean["CPA"] = df_clean["Acquisition_Cost_Clean"] / df_clean["Estimated_Conversions"]
df_clean["Estimated_Revenue"] = df_clean["Acquisition_Cost_Clean"] * (1 + df_clean["ROI"])
df_clean["Estimated_Profit"] = df_clean["Estimated_Revenue"] - df_clean["Acquisition_Cost_Clean"]

# Replace infinite values if any
df_clean.replace([np.inf, -np.inf], np.nan, inplace=True)

df_clean.head()

In [ ]:
# Check cleaned columns
df_clean[[
    "Acquisition_Cost", 
    "Acquisition_Cost_Clean", 
    "CTR_Percent", 
    "Estimated_Conversions", 
    "CPC", 
    "CPA", 
    "Estimated_Revenue", 
    "Estimated_Profit"
]].head()

# Phase 5: Executive Marketing KPIs

These are the most important numbers for a marketing manager or business owner.

In [ ]:
total_campaigns = df_clean["Campaign_ID"].nunique()
total_spend = df_clean["Acquisition_Cost_Clean"].sum()
total_revenue = df_clean["Estimated_Revenue"].sum()
total_profit = df_clean["Estimated_Profit"].sum()
avg_roi = df_clean["ROI"].mean()
avg_conversion_rate = df_clean["Conversion_Rate"].mean()
avg_ctr = df_clean["CTR_Percent"].mean()
total_clicks = df_clean["Clicks"].sum()
total_impressions = df_clean["Impressions"].sum()

kpi_summary = pd.DataFrame({
    "Metric": [
        "Total Unique Campaigns",
        "Total Marketing Spend",
        "Estimated Revenue",
        "Estimated Profit",
        "Average ROI",
        "Average Conversion Rate",
        "Average CTR %",
        "Total Clicks",
        "Total Impressions"
    ],
    "Value": [
        total_campaigns,
        round(total_spend, 2),
        round(total_revenue, 2),
        round(total_profit, 2),
        round(avg_roi, 2),
        round(avg_conversion_rate, 4),
        round(avg_ctr, 2),
        total_clicks,
        total_impressions
    ]
})

kpi_summary

# Phase 6: Channel Performance Analysis

Business question:

> Which marketing channel gives the best ROI, conversion rate, engagement, and profit?

In [ ]:
channel_summary = df_clean.groupby("Channel_Used").agg(
    Campaigns=("Campaign_ID", "nunique"),
    Total_Spend=("Acquisition_Cost_Clean", "sum"),
    Estimated_Revenue=("Estimated_Revenue", "sum"),
    Estimated_Profit=("Estimated_Profit", "sum"),
    Avg_ROI=("ROI", "mean"),
    Avg_Conversion_Rate=("Conversion_Rate", "mean"),
    Avg_CTR_Percent=("CTR_Percent", "mean"),
    Avg_Engagement=("Engagement_Score", "mean"),
    Total_Clicks=("Clicks", "sum"),
    Total_Impressions=("Impressions", "sum")
).reset_index()

channel_summary = channel_summary.sort_values(by="Avg_ROI", ascending=False)
channel_summary

In [ ]:
plt.figure(figsize=(8, 5))
channel_summary.plot(kind="bar", x="Channel_Used", y="Avg_ROI", legend=False)
plt.title("Average ROI by Marketing Channel")
plt.xlabel("Channel")
plt.ylabel("Average ROI")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
channel_summary.plot(kind="bar", x="Channel_Used", y="Avg_Conversion_Rate", legend=False)
plt.title("Average Conversion Rate by Channel")
plt.xlabel("Channel")
plt.ylabel("Average Conversion Rate")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Phase 7: Campaign Goal Analysis

Business question:

> Which campaign objective performs best?

In [ ]:
goal_summary = df_clean.groupby("Campaign_Goal").agg(
    Campaigns=("Campaign_ID", "nunique"),
    Total_Spend=("Acquisition_Cost_Clean", "sum"),
    Estimated_Revenue=("Estimated_Revenue", "sum"),
    Estimated_Profit=("Estimated_Profit", "sum"),
    Avg_ROI=("ROI", "mean"),
    Avg_Conversion_Rate=("Conversion_Rate", "mean"),
    Avg_Engagement=("Engagement_Score", "mean")
).reset_index().sort_values(by="Avg_ROI", ascending=False)

goal_summary

In [ ]:
plt.figure(figsize=(8, 5))
goal_summary.plot(kind="bar", x="Campaign_Goal", y="Avg_ROI", legend=False)
plt.title("Average ROI by Campaign Goal")
plt.xlabel("Campaign Goal")
plt.ylabel("Average ROI")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Phase 8: Customer Segment Analysis

Business question:

> Which customer segment gives the best business value?

In [ ]:
segment_summary = df_clean.groupby("Customer_Segment").agg(
    Campaigns=("Campaign_ID", "nunique"),
    Total_Spend=("Acquisition_Cost_Clean", "sum"),
    Estimated_Revenue=("Estimated_Revenue", "sum"),
    Estimated_Profit=("Estimated_Profit", "sum"),
    Avg_ROI=("ROI", "mean"),
    Avg_Conversion_Rate=("Conversion_Rate", "mean"),
    Avg_CTR_Percent=("CTR_Percent", "mean"),
    Avg_Engagement=("Engagement_Score", "mean")
).reset_index().sort_values(by="Avg_ROI", ascending=False)

segment_summary

In [ ]:
plt.figure(figsize=(8, 5))
segment_summary.plot(kind="bar", x="Customer_Segment", y="Avg_ROI", legend=False)
plt.title("Average ROI by Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Average ROI")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Phase 9: Location and Language Analysis

Business question:

> Which geography and language group performs better?

In [ ]:
location_summary = df_clean.groupby("Location").agg(
    Campaigns=("Campaign_ID", "nunique"),
    Total_Spend=("Acquisition_Cost_Clean", "sum"),
    Estimated_Revenue=("Estimated_Revenue", "sum"),
    Estimated_Profit=("Estimated_Profit", "sum"),
    Avg_ROI=("ROI", "mean"),
    Avg_Conversion_Rate=("Conversion_Rate", "mean")
).reset_index().sort_values(by="Avg_ROI", ascending=False)

location_summary

In [ ]:
language_summary = df_clean.groupby("Language").agg(
    Campaigns=("Campaign_ID", "nunique"),
    Total_Spend=("Acquisition_Cost_Clean", "sum"),
    Estimated_Revenue=("Estimated_Revenue", "sum"),
    Estimated_Profit=("Estimated_Profit", "sum"),
    Avg_ROI=("ROI", "mean"),
    Avg_Conversion_Rate=("Conversion_Rate", "mean")
).reset_index().sort_values(by="Avg_ROI", ascending=False)

language_summary

# Phase 10: Monthly Trend Analysis

Business question:

> Is marketing performance improving or declining month by month?

In [ ]:
monthly_summary = df_clean.groupby("Year_Month").agg(
    Campaigns=("Campaign_ID", "nunique"),
    Total_Spend=("Acquisition_Cost_Clean", "sum"),
    Estimated_Revenue=("Estimated_Revenue", "sum"),
    Estimated_Profit=("Estimated_Profit", "sum"),
    Avg_ROI=("ROI", "mean"),
    Avg_Conversion_Rate=("Conversion_Rate", "mean"),
    Total_Clicks=("Clicks", "sum"),
    Total_Impressions=("Impressions", "sum")
).reset_index()

monthly_summary.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly_summary["Year_Month"], monthly_summary["Avg_ROI"], marker="o")
plt.title("Monthly Average ROI Trend")
plt.xlabel("Month")
plt.ylabel("Average ROI")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly_summary["Year_Month"], monthly_summary["Estimated_Profit"], marker="o")
plt.title("Monthly Estimated Profit Trend")
plt.xlabel("Month")
plt.ylabel("Estimated Profit")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Phase 11: Relationship Analysis

Business question:

> Which metrics are related to ROI and conversion rate?

In [ ]:
numeric_cols = [
    "Conversion_Rate",
    "Acquisition_Cost_Clean",
    "ROI",
    "Clicks",
    "Impressions",
    "Engagement_Score",
    "CTR_Percent",
    "Estimated_Conversions",
    "CPC",
    "CPA",
    "Estimated_Revenue",
    "Estimated_Profit"
]

correlation_matrix = df_clean[numeric_cols].corr()
correlation_matrix

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df_clean["Clicks"], df_clean["ROI"], alpha=0.2)
plt.title("Clicks vs ROI")
plt.xlabel("Clicks")
plt.ylabel("ROI")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df_clean["Engagement_Score"], df_clean["Conversion_Rate"], alpha=0.2)
plt.title("Engagement Score vs Conversion Rate")
plt.xlabel("Engagement Score")
plt.ylabel("Conversion Rate")
plt.tight_layout()
plt.show()

# Phase 12: Campaign Performance Segmentation using K-Means

Now we create AI-based campaign groups.

This helps the business identify:

- High ROI campaigns
- High engagement campaigns
- Costly campaigns
- Low performance campaigns

In [ ]:
cluster_features = [
    "Conversion_Rate",
    "Acquisition_Cost_Clean",
    "ROI",
    "Clicks",
    "Impressions",
    "Engagement_Score",
    "CTR_Percent",
    "Estimated_Profit"
]

X_cluster = df_clean[cluster_features].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_clean["Campaign_Cluster"] = kmeans.fit_predict(X_scaled)

cluster_summary = df_clean.groupby("Campaign_Cluster").agg(
    Campaigns=("Campaign_ID", "nunique"),
    Avg_ROI=("ROI", "mean"),
    Avg_Conversion_Rate=("Conversion_Rate", "mean"),
    Avg_Spend=("Acquisition_Cost_Clean", "mean"),
    Avg_Engagement=("Engagement_Score", "mean"),
    Avg_Profit=("Estimated_Profit", "mean")
).reset_index().sort_values(by="Avg_ROI", ascending=False)

cluster_summary

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df_clean["ROI"], df_clean["Conversion_Rate"], c=df_clean["Campaign_Cluster"], alpha=0.3)
plt.title("Campaign Clusters: ROI vs Conversion Rate")
plt.xlabel("ROI")
plt.ylabel("Conversion Rate")
plt.tight_layout()
plt.show()

## How to Read Clusters

After running the above cell:

- Cluster with high ROI + high conversion = **Best campaigns**
- Cluster with high spend + low ROI = **Wasteful campaigns**
- Cluster with medium ROI = **Optimization opportunity**
- Cluster with high engagement but low conversion = **Message/offer issue**

---

# Phase 13: Machine Learning Model - Predict High ROI Campaigns

We will create a simple AI model.

### Target
`High_ROI = 1` if ROI is in the top 25%  
`High_ROI = 0` otherwise

### Why this is useful
Marketing teams can predict whether a campaign is likely to become high ROI before scaling budget.

In [ ]:
# Create target column
roi_threshold = df_clean["ROI"].quantile(0.75)
df_clean["High_ROI"] = (df_clean["ROI"] >= roi_threshold).astype(int)

print("High ROI threshold:", round(roi_threshold, 2))
df_clean["High_ROI"].value_counts(normalize=True)

In [ ]:
# Features for ML model
feature_cols = [
    "Target_Audience",
    "Campaign_Goal",
    "Duration",
    "Channel_Used",
    "Location",
    "Language",
    "Customer_Segment",
    "Clicks",
    "Impressions",
    "Engagement_Score",
    "Conversion_Rate",
    "Acquisition_Cost_Clean",
    "CTR_Percent"
]

X = df_clean[feature_cols]
y = df_clean["High_ROI"]

categorical_features = [
    "Target_Audience",
    "Campaign_Goal",
    "Duration",
    "Channel_Used",
    "Location",
    "Language",
    "Customer_Segment"
]

numeric_features = [
    "Clicks",
    "Impressions",
    "Engagement_Score",
    "Conversion_Rate",
    "Acquisition_Cost_Clean",
    "CTR_Percent"
]

# To keep training fast for beginners, use a sample if the dataset is very large
sample_size = min(60000, len(df_clean))
sample_data = df_clean.sample(sample_size, random_state=42)

X_sample = sample_data[feature_cols]
y_sample = sample_data["High_ROI"]

X_train, X_test, y_train, y_test = train_test_split(
    X_sample, 
    y_sample, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_sample
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ]
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual Low ROI", "Actual High ROI"],
    columns=["Predicted Low ROI", "Predicted High ROI"]
)

cm_df

# Phase 14: Feature Importance

This helps explain which factors are most important for predicting high ROI campaigns.

In [ ]:
# Get feature names after one-hot encoding
encoded_cat_features = model.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out(categorical_features)
all_feature_names = list(encoded_cat_features) + numeric_features

importances = model.named_steps["classifier"].feature_importances_

feature_importance = pd.DataFrame({
    "Feature": all_feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

feature_importance.head(20)

In [ ]:
top_features = feature_importance.head(15)

plt.figure(figsize=(10, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Top 15 Important Features for High ROI Prediction")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Phase 15: Senior Data Scientist Insights

Run this cell to generate automatic business insights from your dataset.

In [ ]:
best_channel = channel_summary.sort_values("Avg_ROI", ascending=False).iloc[0]
best_segment = segment_summary.sort_values("Avg_ROI", ascending=False).iloc[0]
best_goal = goal_summary.sort_values("Avg_ROI", ascending=False).iloc[0]
best_location = location_summary.sort_values("Avg_ROI", ascending=False).iloc[0]
best_language = language_summary.sort_values("Avg_ROI", ascending=False).iloc[0]

print("Senior Marketing Insights")
print("=" * 60)
print(f"1. Best ROI Channel: {best_channel['Channel_Used']} with average ROI {best_channel['Avg_ROI']:.2f}")
print(f"2. Best Customer Segment: {best_segment['Customer_Segment']} with average ROI {best_segment['Avg_ROI']:.2f}")
print(f"3. Best Campaign Goal: {best_goal['Campaign_Goal']} with average ROI {best_goal['Avg_ROI']:.2f}")
print(f"4. Best Location: {best_location['Location']} with average ROI {best_location['Avg_ROI']:.2f}")
print(f"5. Best Language: {best_language['Language']} with average ROI {best_language['Avg_ROI']:.2f}")
print()
print("Recommended Business Actions")
print("=" * 60)
print("- Increase budget for the highest ROI channel.")
print("- Create separate targeting strategy for the best customer segment.")
print("- Reduce or optimize campaigns with high CPA and low ROI.")
print("- Use high-engagement campaigns for retargeting.")
print("- Before scaling any campaign, use the ML model to estimate High ROI probability.")

# Phase 16: Save Cleaned Dataset for Power BI

This cleaned file can be imported directly into Power BI.

In [ ]:
output_file = "Social_Media_Advertising_Cleaned_For_PowerBI.csv"
df_clean.to_csv(output_file, index=False)

print("Cleaned dataset saved successfully:", output_file)

# Final Project Summary

## Project Title
**Social Media Advertising Performance Analytics using Python, Machine Learning and Power BI**

## What We Did
- Loaded and understood 300,000 advertising campaign records
- Cleaned acquisition cost and date columns
- Created marketing KPIs: CTR, CPC, CPA, estimated conversions, revenue, and profit
- Analyzed campaign performance by channel, goal, segment, location, and language
- Built K-Means campaign segmentation
- Built a Random Forest model to predict high ROI campaigns

## Business Value
This project helps marketing teams:

- Find best-performing channels
- Improve marketing budget allocation
- Identify high ROI customer segments
- Reduce wasteful ad spend
- Predict campaign success using AI

## Power BI Dashboard Pages You Can Build
1. Executive Summary
2. Channel Performance
3. Campaign Goal Analysis
4. Customer Segment Analysis
5. Location and Language Analysis
6. AI Campaign Cluster Analysis
7. High ROI Prediction Dashboard

---  
Notebook prepared in a simple **Senior Data Scientist + AI Engineer** workflow. 🚀